In [1]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import shared.utils as su

In [9]:
# Load all chiral verb pairs

In [26]:
from utils.gemini_utils import GeminiWrapper

model_key = "gemini-2.0-flash"
vlm = GeminiWrapper(model_key=model_key, fps=1.)
vlm.forward_text_only("Add 23 + 45")

Loading gemini-2.0-flash with FPS=1.0...........................................  


'23 + 45 = 68\n'

In [2]:
data_root = "/scratch/shared/beegfs/piyush/datasets"

file = f"{data_root}/Tarsier2/ActivityNet/metadata.json"
data = su.io.load_json(file)
len(data)

35960

In [3]:
data[0]

{'messages': [{'content': [{'type': 'video',
     'video': {'video_file': 'ActivityNet/videos/v_QOlSCBRmfWY.mp4',
      'start_time': 0.8299999833,
      'end_time': 19.8600006104,
      'start_frame': None,
      'end_frame': None}},
    {'type': 'text', 'text': 'Describe the video in detail.'}],
   'role': 'user'},
  {'content': [{'text': 'A woman stands in a wooden-floored room with large windows and a wooden door, wearing a black sports bra and shorts. She begins by walking to the left side of the room. She then performs a series of dance movements, including a jump with arms extended, a squat, and a spin on one foot. She transitions into a tree pose, balancing on one foot with arms raised. She then moves into a floor routine, starting with a roll and transitioning into various poses, including a backbend and a split.',
     'type': 'text'}],
   'role': 'assistant'}],
 'dataset': 'ActivityNet',
 'task': 'video/caption',
 'idx': 'ActivityNet_0'}

In [6]:
file = f"{data_root}/MSRVTT/annotation/msrvtt_test_1k.json"
df = pd.DataFrame(su.io.load_json(file))
len(df)

1000

In [8]:
captions = df.caption.tolist()
len(captions)

1000

In [27]:
file_path = "./chiral_score_prompt.txt"
with open(file_path, 'r', encoding='utf-8') as file:
    prompt_template = file.read()

In [28]:
caption = captions[1]
prompt = prompt_template.replace("<INSERT SENTENCE HERE>", caption)
print(caption)

baseball player hits ball


In [29]:
answer = vlm.forward_text_only(prompt)
print(answer)

```json
{
  "sentence": "baseball player hits ball",
  "verb_phrases": [
    {
      "verb_phrase": "hits ball",
      "temporal_antonym": null,
      "reasoning": "Rewinding a video of hitting a ball shows the ball flying back toward the bat and making contact. This isn't a readily nameable action, such as 'catching' because the bat is still involved and it's an unnatural movement pattern. There's no corresponding verb phrase for this.",
      "chiral_score": 0.2
    }
  ],
  "overall_chiral_score": 0.2
}
```


**Run over all captions**

In [30]:
answers = []
for c in su.log.tqdm_iterator(captions, desc="Generating annotations"):
    prompt = prompt_template.replace("<INSERT SENTENCE HERE>", c)
    answer = vlm.forward_text_only(prompt)
    answers.append(answer)
len(answers)

Generating annotations:   0%|          | 0/1000 [00:00<?, ?it/s]

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}

In [59]:
su.io.save_jsonl(answers, "chiral_scores-msrvtt.jsonl")

In [32]:
subdf = df.iloc[:len(answers)]
subdf.shape

(795, 9)

In [38]:
import json
import re


def parse_chiral_response(raw: str) -> dict:
    """Parse a chiral score JSON response from an LLM.

    Handles:
    - Clean JSON wrapped in ```json ... ```
    - JSON with leading/trailing explanation text
    - No JSON at all (returns default 0.0 score structure)
    """
    # Try to extract a fenced JSON block anywhere in the response
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", raw, re.DOTALL)
    if match:
        return json.loads(match.group(1))

    # Try to find a raw JSON object (no fences)
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass

    # No valid JSON found — return default structure
    return {
        "sentence": None,
        "verb_phrases": [],
        "overall_chiral_score": 0.0,
    }


parse_chiral_response(answers[10])

{'sentence': 'a man extinguishes a fire outside',
 'verb_phrases': [{'verb_phrase': 'extinguishes a fire',
   'temporal_antonym': 'starting a fire',
   'reasoning': 'The reversed footage would show the fire growing from nothing, smoke coalescing, and fuel igniting; a near mirror image of starting a fire.',
   'chiral_score': 0.9}],
 'overall_chiral_score': 0.9}

In [40]:
refined_answers = []
failed = 0
for a in answers:
    try:
        b = parse_chiral_response(a)
    except:
        print(a)
        print('-' * 100)
        failed += 1
        continue
    refined_answers.append(b)
print("Number of failed samples: ", failed)

len(refined_answers)

Number of failed samples:  0


795

In [45]:
avg_score = []
for a in refined_answers:
    verb_phrases = a['verb_phrases']
    c = a['overall_chiral_score']
    avg_score.append(float(c))
np.mean(avg_score)

0.40352830188679245

### ActivityNet

In [50]:
file = f"{data_root}/ActivityNetCaptions/val_1.json"
data = su.io.load_json(file)
len(data)

4917

In [52]:
data['v_uqiMw7tQ1Cc']

{'duration': 55.15,
 'timestamps': [[0.28, 55.15], [13.79, 54.32]],
 'sentences': ['A weight lifting tutorial is given.',
  '  The coach helps the guy in red with the proper body placement and lifting technique.']}

In [57]:
captions = []
for k, d in data.items():
    captions.extend(d['sentences'])
len(captions)

17505

In [58]:
# Sample K captions
sampled_captions = np.random.choice(captions, 500)
len(sampled_captions)

500

In [60]:
answers = []
for c in su.log.tqdm_iterator(sampled_captions, desc="Generating annotations"):
    prompt = prompt_template.replace("<INSERT SENTENCE HERE>", c)
    answer = vlm.forward_text_only(prompt)
    answers.append(answer)
len(answers)

Generating annotations:   0%|          | 0/500 [00:00<?, ?it/s]

500

In [61]:
su.io.save_jsonl(answers, "chiral_scores-activitynet.jsonl")

In [62]:
refined_answers = []
failed = 0
for a in answers:
    try:
        b = parse_chiral_response(a)
    except:
        print(a)
        print('-' * 100)
        failed += 1
        continue
    refined_answers.append(b)
print("Number of failed samples: ", failed)

len(refined_answers)

```json
{
  "sentence": "`A title screen appears with \"Livermore Dog Grooming Wine Country Pet Spa Meet: 'Millie'".`",
  "verb_phrases": [
    {
      "verb_phrase": "appears",
      "temporal_antonym": "disappears",
      "reasoning": "Rewinding a video of something appearing (e.g., on a screen) shows it vanishing, which is well described by the verb 'disappears'.",
      "chiral_score": 0.9
    }
  ],
  "overall_chiral_score": 0.9
}
```
----------------------------------------------------------------------------------------------------
Number of failed samples:  1


499

In [63]:
avg_score = []
for a in refined_answers:
    verb_phrases = a['verb_phrases']
    c = a['overall_chiral_score']
    avg_score.append(float(c))
np.mean(avg_score)

0.5584103874415498

In [68]:
count = 0
for x in refined_answers:
    for y in x['verb_phrases']:
        if y['temporal_antonym'] is not None:
            count += 1
            break
print(count / len(refined_answers))

0.7414829659318637


### DiDeMo

In [70]:
file = f"{data_root}/DiDeMo/raw_data/didemo_test.json"
data = su.io.load_json(file)
df = pd.DataFrame(data)
captions = df['description'].tolist()
df.shape

(4021, 7)

In [72]:
sampled_captions = np.random.choice(df['description'], 500)
len(sampled_captions)

500

In [73]:
answers = []
for c in su.log.tqdm_iterator(sampled_captions, desc="Generating annotations"):
    prompt = prompt_template.replace("<INSERT SENTENCE HERE>", c)
    answer = vlm.forward_text_only(prompt)
    answers.append(answer)
len(answers)

Generating annotations:   0%|          | 0/500 [00:00<?, ?it/s]

500

In [74]:
su.io.save_jsonl(answers, "chiral_scores-didemo.jsonl")

In [75]:
refined_answers = []
failed = 0
for a in answers:
    try:
        b = parse_chiral_response(a)
    except:
        print(a)
        print('-' * 100)
        failed += 1
        continue
    refined_answers.append(b)
print("Number of failed samples: ", failed)

len(refined_answers)

Number of failed samples:  0


500

In [76]:
avg_score = []
for a in refined_answers:
    verb_phrases = a['verb_phrases']
    c = a['overall_chiral_score']
    avg_score.append(float(c))
np.mean(avg_score)

0.6888500000000001

### MSVD

In [79]:
df = pd.DataFrame(su.io.load_json(f"{data_root}/MSVD/msvd_test.json"))
captions = df.caption.apply(lambda x: x[0]).tolist()

len(df)

670

In [85]:
captions = df.caption.apply(lambda x: x[0]).tolist()
sampled_captions = np.random.choice(captions, 500)
len(sampled_captions)

500

In [86]:
answers = []
for c in su.log.tqdm_iterator(sampled_captions, desc="Generating annotations"):
    prompt = prompt_template.replace("<INSERT SENTENCE HERE>", c)
    answer = vlm.forward_text_only(prompt)
    answers.append(answer)
len(answers)

Generating annotations:   0%|          | 0/500 [00:00<?, ?it/s]

500

In [87]:
su.io.save_jsonl(answers, "chiral_scores-msvd.jsonl")

In [88]:
refined_answers = []
failed = 0
for a in answers:
    try:
        b = parse_chiral_response(a)
    except:
        print(a)
        print('-' * 100)
        failed += 1
        continue
    refined_answers.append(b)
print("Number of failed samples: ", failed)

len(refined_answers)

```json
{
  "sentence": "a man has his arm around a woman and they are walking in the woods",
  "verb_phrases": [
    {
      "verb_phrase": "has his arm around a woman",
      "temporal_antonym": null,
      "reasoning": "Rewinding the video shows the man's arm moving away from the woman. This could vaguely be interpreted as 'removing his arm', but doesn't have a crisp, clear action associated with it. It's less about a distinct action and more about releasing a hold. ",
      "chiral_score": 0.3
    },
    {
      "verb_phrase": "walking in the woods",
      "temporal_antonym": "walking out of the woods" or "walking backwards in the woods",
      "reasoning": "Rewinding a video of walking shows the people walking backwards. While 'walking backwards' is a valid action, the context of 'in the woods' suggests 'walking out of the woods' as a more relevant, if less perfect, temporal antonym. Also walking backwards can be considered a reverse of walking forwards. However, depending on the 

499

In [89]:
avg_score = []
for a in refined_answers:
    verb_phrases = a['verb_phrases']
    c = a['overall_chiral_score']
    avg_score.append(float(c))
np.mean(avg_score)

0.4928977955911823

### Check for Negation

In [92]:
def load_captions(ds):
    data_root = "/scratch/shared/beegfs/piyush/datasets"
    if ds == "activitynet":
        file = f"{data_root}/ActivityNetCaptions/val_1.json"
        data = su.io.load_json(file)
        captions = []
        for k, d in data.items():
            captions.extend(d['sentences'])
    elif ds == "msrvtt":
        file = f"{data_root}/MSRVTT/annotation/msrvtt_test_1k.json"
        df = pd.DataFrame(su.io.load_json(file))
        captions = df.caption.tolist()
    elif ds == "didemo":
        file = f"{data_root}/DiDeMo/raw_data/didemo_test.json"
        data = su.io.load_json(file)
        df = pd.DataFrame(data)
        captions = df['description'].tolist()
    elif ds == "msvd":
        df = pd.DataFrame(su.io.load_json(f"{data_root}/MSVD/msvd_test.json"))
        captions = df.caption.apply(lambda x: x[0]).tolist()
    else:
        raise ValueError
    return captions


for ds in ['activitynet', 'didemo', 'msrvtt', 'msvd']:
    captions = load_captions(ds)

    # Check negation
    count = sum([" not " in x for x in captions])
    print(f"{ds}: {np.round(count * 100./ len(captions), 2)}")

activitynet: 0.25
didemo: 0.32
msrvtt: 0.2
msvd: 0.0
